<a href="https://colab.research.google.com/github/kady05naik/PySpark/blob/main/transformations_based_on_dataframe_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformations on Data Frame 13

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

**As we will be performing operations related to window we have to import window**

In [18]:
from pyspark.sql.window import Window

In [3]:
spark=SparkSession.builder \
  .appName('MyApp') \
  .getOrCreate()

In [5]:
simpleData = (("James", "Sales", 3000),
    ("Michael", "Sales", 4600),
    ("Robert", "Sales", 4100),
    ("Maria", "Finance", 3000),
    ("James", "Sales", 3000),
    ("Scott", "Finance", 3300),
    ("Jen", "Finance", 3900),
    ("Jeff", "Marketing", 3000),
    ("Kumar", "Marketing", 2000),
    ("Saif", "Sales", 4100)
)


In [7]:
schema=StructType([
    StructField('employee_name', StringType(), True),
    StructField('department', StringType(), True),
    StructField('salary', IntegerType(), True)
])

In [12]:
df=spark.createDataFrame(simpleData, schema)



---



**1) Create a row_number on salary basis**

In [22]:
win_sal= Window.orderBy(col('salary').desc())
df.withColumn("row_number", row_number().over(win_sal)) \
  .show()

+-------------+----------+------+----------+
|employee_name|department|salary|row_number|
+-------------+----------+------+----------+
|      Michael|     Sales|  4600|         1|
|       Robert|     Sales|  4100|         2|
|         Saif|     Sales|  4100|         3|
|          Jen|   Finance|  3900|         4|
|        Scott|   Finance|  3300|         5|
|        James|     Sales|  3000|         6|
|        Maria|   Finance|  3000|         7|
|        James|     Sales|  3000|         8|
|         Jeff| Marketing|  3000|         9|
|        Kumar| Marketing|  2000|        10|
+-------------+----------+------+----------+





---



**2) Find the lowest salary of the employee**

In [30]:
win_sal=Window.orderBy(col('salary'))
df.withColumn('dense_rank', dense_rank().over(win_sal)) \
  .filter(col('dense_rank')==1) \
  .select('employee_name','department', 'salary') \
  .show()

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|        Kumar| Marketing|  2000|
+-------------+----------+------+





---



**3) Find the second highest salary of the employee**

In [31]:
win_sal=Window.orderBy(col('salary').desc())

df.withColumn("dense_rank",dense_rank().over(win_sal)) \
  .filter(col('dense_rank')==2) \
  .select('employee_name','department', 'salary') \
  .show()

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|       Robert|     Sales|  4100|
|         Saif|     Sales|  4100|
+-------------+----------+------+





---



**4) Find the second highest salary department-wise**

In [34]:
win_sal=Window.partitionBy(col('department')) \
  .orderBy(col('salary').desc())

df.withColumn('dense_rank', dense_rank().over(win_sal)) \
  .filter(col('dense_rank')==2) \
  .select('employee_name','department','salary') \
  .show()

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|        Scott|   Finance|  3300|
|        Kumar| Marketing|  2000|
|       Robert|     Sales|  4100|
|         Saif|     Sales|  4100|
+-------------+----------+------+





---



**5) Find the second highest salary of the employee whose department is Sales**

In [39]:
sales_df=df.filter(col('department')=='Sales')
win_sal=Window.orderBy(col('salary').desc())
sales_df.withColumn('dense_rank', dense_rank().over(win_sal)) \
  .filter(col('dense_rank')==2) \
  .show()

+-------------+----------+------+----------+
|employee_name|department|salary|dense_rank|
+-------------+----------+------+----------+
|       Robert|     Sales|  4100|         2|
|         Saif|     Sales|  4100|         2|
+-------------+----------+------+----------+





---



**6) Create a row_number starting with sequence number 0 with no order**

row_number() in Spark always starts from 1, not 0.
To make it start from 0, subtract 1.

Since you asked for no order, use any dummy order expression.

In [40]:
w=Window.orderBy(lit(1))
df.withColumn('row_number', row_number().over(w)-1).show()

+-------------+----------+------+----------+
|employee_name|department|salary|row_number|
+-------------+----------+------+----------+
|        James|     Sales|  3000|         0|
|      Michael|     Sales|  4600|         1|
|       Robert|     Sales|  4100|         2|
|        Maria|   Finance|  3000|         3|
|        James|     Sales|  3000|         4|
|        Scott|   Finance|  3300|         5|
|          Jen|   Finance|  3900|         6|
|         Jeff| Marketing|  3000|         7|
|        Kumar| Marketing|  2000|         8|
|         Saif|     Sales|  4100|         9|
+-------------+----------+------+----------+





---



**7) Create a row_number starting with sequence number 1 with no order**

Again, with no meaningful order, sequence assignment may vary between runs.

In [41]:
win=Window.orderBy(lit(1))
df.withColumn('row_number', row_number().over(w)).show()

+-------------+----------+------+----------+
|employee_name|department|salary|row_number|
+-------------+----------+------+----------+
|        James|     Sales|  3000|         1|
|      Michael|     Sales|  4600|         2|
|       Robert|     Sales|  4100|         3|
|        Maria|   Finance|  3000|         4|
|        James|     Sales|  3000|         5|
|        Scott|   Finance|  3300|         6|
|          Jen|   Finance|  3900|         7|
|         Jeff| Marketing|  3000|         8|
|        Kumar| Marketing|  2000|         9|
|         Saif|     Sales|  4100|        10|
+-------------+----------+------+----------+



In [42]:
spark.stop()